# Assignment 3
## Econ 8310 - Business Forecasting

For homework assignment 3, you will work with our baseball pitch data.

You must create a custom data loader as described in the first week of neural network lectures to load the baseball videos [2 points]
You must create a working and trained neural network (any network focused on the baseball pitch videos will do) using only pytorch [2 points]
You must store your weights and create an import script so that I can evaluate your model without training it [2 points]

Submit your forked repository URL here! :) I'll be manually grading this assignment.

In [1]:
# For reading data
import pandas as pd
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# For visualizing
import plotly.express as px

# For model building
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class CustomMNIST(Dataset):
    def __init__(self, url):
        # read in our raw data from the hosting URL
        self.raw_data = pd.read_csv(url)

    # return the length of the complete data set
    #   to be used in internal calculations for pytorch
    def __len__(self):
        return self.raw_data.shape[0]

    # retrieve a single record based on index position `idx`
    def __getitem__(self, idx):
        # extract the image separate from the label
        image = self.raw_data.iloc[idx, 1:].values.reshape(1, 28, 28)
        # Specify dtype to align with default dtype used by weight matrices
        image = torch.tensor(image, dtype=torch.float32)
        # extract the label
        label = self.raw_data.iloc[idx, 0]

        # return the image and its corresponding label
        return image, label
        

In [ ]:
# Load our data into memory
train_data = CustomMNIST("https://github.com/dustywhite7/
Econ8310/raw/master/DataSets/mnistTrain.csv")
test_data = CustomMNIST("https://github.com/dustywhite7/
Econ8310/raw/master/DataSets/mnistTest.csv")

# Create data feed pipelines for modeling
train_dataloader = DataLoader(train_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

In [ ]:
import torch
from torch.utils.data import Dataset
import xml.etree.ElementTree as ET
import cv2
import numpy as np

class BaseballPitchDataset(Dataset):
    def __init__(self, xml_path, transform=None, frames_per_clip=16):
        self.transform = transform
        self.frames_per_clip = frames_per_clip
        self.samples = []  # list of (video_path, label)
        
        # Define your pitch classes — update to match your XML labels
        self.classes = ['fastball', 'curveball', 'slider', 'changeup']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        
        # Parse XML
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        # ⚠️ Update these tag names to match YOUR XML structure
        for pitch in root.findall('pitch'):
            video_path = pitch.find('video_path').text
            label = pitch.find('pitch_type').text
            self.samples.append((video_path, self.class_to_idx[label]))

    def __len__(self):
        return len(self.samples)

    def _load_video(self, path):
        cap = cv2.VideoCapture(path)
        frames = []
        while len(frames) < self.frames_per_clip:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, (112, 112))
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)
        cap.release()
        
        # Pad if video is shorter than frames_per_clip
        while len(frames) < self.frames_per_clip:
            frames.append(frames[-1])
        
        # Shape: (T, H, W, C) → (C, T, H, W) for PyTorch
        video = np.stack(frames[:self.frames_per_clip])
        video = torch.tensor(video, dtype=torch.float32).permute(3, 0, 1, 2)
        video = video / 255.0  # normalize to [0, 1]
        return video

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        video = self._load_video(video_path)
        if self.transform:
            video = self.transform(video)
        return video, label

In [ ]:
import torch
import torch.nn as nn

class PitchClassifier(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        # 3D conv to capture spatial + temporal features
        self.features = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=(3,3,3), padding=1),
            nn.ReLU(),
            nn.MaxPool3d((1,2,2)),
            
            nn.Conv3d(32, 64, kernel_size=(3,3,3), padding=1),
            nn.ReLU(),
            nn.MaxPool3d((2,2,2)),
            
            nn.Conv3d(64, 128, kernel_size=(3,3,3), padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool3d((2,4,4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 2 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


In [ ]:

# For reading data
from torch.utils.data import DataLoader

# For visualizing
import plotly.express as px

# For model building
import torch
import torch.nn as nn
import torch.nn.functional as F

# Import helpers
import nnhelpers as nnh
# Still loading the same data as last week!
# Load our data into memory
train_data = nnh.CustomMNIST("https://github.com/dustywhite7/
Econ8310/raw/master/DataSets/mnistTrain.csv")
test_data = nnh.CustomMNIST("https://github.com/dustywhite7/
Econ8310/raw/master/DataSets/mnistTest.csv")

# Create data feed pipelines for modeling
train_dataloader = DataLoader(train_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)
class LeNet(nn.Module):

    def __init__(self):
        super(LeNet, self).__init__()
        # 6 output channels, 5x5 square convolution
        # kernel
        self.conv1 = nn.LazyConv2d(6, 5, padding=2)
        self.conv2 = nn.LazyConv2d(16, 5)
        # an affine operation: y = Wx + b
        self.fc1 = nn.LazyLinear(120)  
        self.fc2 = nn.LazyLinear(84)
        self.fc3 = nn.LazyLinear(10)

class LeNet(nn.Module):
    ...

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is square, you can specify with a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        # flatten all dimensions except the batch dimension
        x = torch.flatten(x, 1) 
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Create a model instance, pass the model to GPU
model = LeNet().to('cuda')

model = nnh.train_net(model, train_dataloader, 
        test_dataloader, epochs = 5, learning_rate = 1e-3,
        batch_size=64
        )

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from dataset import BaseballPitchDataset
from model import PitchClassifier

# Config
XML_PATH = "pitches.xml"
WEIGHTS_PATH = "pitch_model_weights.pth"
EPOCHS = 10
BATCH_SIZE = 8
LR = 1e-4
NUM_CLASSES = 4

dataset = BaseballPitchDataset(XML_PATH)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE)

model = PitchClassifier(num_classes=NUM_CLASSES)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    for videos, labels in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(videos), labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{EPOCHS} — loss: {loss.item():.4f}")

# Save weights only (not the full model)
torch.save(model.state_dict(), WEIGHTS_PATH)
print(f"Weights saved to {WEIGHTS_PATH}")




In [ ]:
import torch
from model import PitchClassifier

def load_model(weights_path="pitch_model_weights.pth", num_classes=4):
    model = PitchClassifier(num_classes=num_classes)
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    return model

if __name__ == "__main__":
    model = load_model()
    print("Model loaded successfully!")
    # Quick sanity check with a dummy input: (batch, C, T, H, W)
    dummy = torch.randn(1, 3, 16, 112, 112)
    with torch.no_grad():
        out = model(dummy)
    print(f"Output shape: {out.shape}")  # should be (1, 4)